In [2]:
# 连接sqlite
# pip install sqlalchemy
from sqlalchemy import create_engine

engine=create_engine("sqlite:///test.db")

with engine.connect() as conn:
    print("success")

success


In [5]:
# sqlite3原生连接
import sqlite3

conn=sqlite3.connect("test.db")

#连接到内存数据库,程序退出后数据消失
# conn_memory=sqlite3.connect(":memory"")
conn.close()

In [15]:
# read_sql 读取数据
import pandas as pd
from sqlalchemy import create_engine

engine=create_engine("sqlite:///test.db")

df=pd.read_sql("employees",con=engine)
print(df)
print(df.head())
print(f"count:{len(df)}")

   c
0  1
   c
0  1
count:1


In [29]:
# 执行查询
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

# DB-API 风格（使用 ?）
df = pd.read_sql(
    "SELECT * FROM employees WHERE   c > ?",
    con=engine,
    params=(0,)   # params 是列表，按位置替换 ?。也可以写成[(0,)]
)
print(df)

# SQLAlchemy 命名参数风格 :paraName
df=pd.read_sql("select * from employees  where c> :a",con=engine,params={"a":0})
print(df)

   c
0  1
   c
0  1


In [35]:
# 设置索引
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

df = pd.read_sql(
    "SELECT * FROM employees ",
    con=engine,
    index_col="c"
)
print(df)

# 符合索引
df = pd.read_sql(
    "SELECT * FROM employees ",
    con=engine,
    index_col=["c","name"]
)
print(df)

  name   o
c         
1   aa  11
2   bb  22
         o
c name    
1 aa    11
2 bb    22


In [38]:
# 解析日期列
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

df = pd.read_sql(
    "SELECT * FROM employees ",
    con=engine,
    parse_dates=["time"]
)
print(df.dtypes)
print(df)

# 指定日期格式
df = pd.read_sql(
    "SELECT * FROM employees ",
    con=engine,
    parse_dates={"time":"%Y%m%d"})
print(df.dtypes)
print(df)

c                int64
name               str
o                  str
time    datetime64[us]
dtype: object
   c name   o       time
0  1   aa  11 2026-10-23
1  2   bb  22 2026-10-23
c                int64
name               str
o                  str
time    datetime64[us]
dtype: object
   c name   o       time
0  1   aa  11 2026-10-23
1  2   bb  22 2026-10-23


In [44]:
# 分块读取大数据
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

chunks  = pd.read_sql(
    "SELECT * FROM large_table ",
    con=engine,
    chunksize=50
)
result=[]
for i,chunk in enumerate(chunks ):
    # 过滤数据
    p=chunk[chunk["id"]!=None]
    result.append(p)
    
# 合并数据
d=pd.concat(result,ignore_index=True)
print(d)

       id
0       1
1       2
2       3
3       4
4       5
..    ...
995   996
996   997
997   998
998   999
999  1000

[1000 rows x 1 columns]


In [47]:
# read_sql_query 只执行查询语句
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

d  = pd.read_sql_query(
    "SELECT * FROM large_table ",
    con=engine
)
print(d)

       id
0       1
1       2
2       3
3       4
4       5
..    ...
995   996
996   997
997   998
998   999
999  1000

[1000 rows x 1 columns]


In [50]:
# 读取整张表
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

d  = pd.read_sql_table(
    "large_table",
    con=engine,
    columns=["id"] # 指定读取的列
)
print(d)

d  = pd.read_sql_table(
    "large_table",
    con=engine,
    columns=["id"], # 指定读取的列
    schema="main" # 指定schema
)
print(d)

       id
0       1
1       2
2       3
3       4
4       5
..    ...
995   996
996   997
997   998
998   999
999  1000

[1000 rows x 1 columns]
       id
0       1
1       2
2       3
3       4
4       5
..    ...
995   996
996   997
997   998
998   999
999  1000

[1000 rows x 1 columns]


In [3]:
# dataFrame写入数据库
from sqlalchemy import create_engine
import pandas as pd

engine=create_engine("sqlite:///test.db")

df = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "department": ["IT", "HR", "IT"],
    "salary": [12000, 8000, 15000]
})

df.to_sql("test",
          con=engine,
          if_exists="replace",
          index=False)

# 追加数据
newData=pd.DataFrame({
    "name": ["张三2"],
    "department": ["IT"],
    "salary": [12000]
})
df.to_sql("test",
          con=engine,
          if_exists="append",
          index=False)

result = pd.read_sql("SELECT * FROM test", con=engine)
print(result)

  name department  salary
0   张三         IT   12000
1   李四         HR    8000
2   王五         IT   15000
3   张三         IT   12000
4   李四         HR    8000
5   王五         IT   15000


In [10]:
# 指定列的数据类型
import pandas as pd
from sqlalchemy import create_engine, Integer, String, Float, DateTime

engine = create_engine("sqlite:///test.db")

df = pd.DataFrame({
    "department": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "salary": [92, 88, 95],
})

df.to_sql("test",
          con=engine,
          if_exists="replace",
          index=False,
         dtype={
             "name":String(40),
             "department":String(40),
             "salary":Integer
         })
result = pd.read_sql("SELECT * FROM test", con=engine)
print(result)

  department     name  salary
0          1    Alice      92
1          2      Bob      88
2          3  Charlie      95


In [ ]:
# DB-API连接池问题
import sqlite3
import pandas as pd

with sqlite3.connect("test.db") as conn:
    df=